# Pipeline completo: Cuento ilustrado (Llama + SD 1.5)

Genera un **cuento infantil ilustrado** de punta a punta:

1. **Llama 3.1-8B + LoRA** genera el cuento condicionado por temática.
2. El mismo Llama **extrae una escena visual** del cuento (una frase corta).
3. **SD 1.5 + LoRA** genera la ilustración con: temática (estilo) + escena (relevancia).
4. Se muestra el cuento + su ilustración juntos.

> Pensado para Colab Pro+ (A100). Los dos modelos caben en memoria: Llama en 4-bit (~5 GB) + SD en fp16 (~3.5 GB).


## Paso 0 — Setup

In [ ]:
import os, sys
EN_COLAB = "google.colab" in sys.modules
print("En Colab:", EN_COLAB)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "Sin GPU"

if EN_COLAB:
    !pip install -q -U transformers peft accelerate bitsandbytes diffusers safetensors
    !pip uninstall -q -y torchao
    from google.colab import drive
    drive.mount("/content/drive")

# Si usas Llama (gated), pon tu token:
from huggingface_hub import login
_hf_token = os.environ.get("HF_TOKEN", "")
if _hf_token:
    login(_hf_token)
else:
    print("HF_TOKEN no definido: necesario para Llama-3.1 (gated). Defínelo en los Secrets de Colab; nunca lo pegues en el código.")


## Paso 1 — Configuración de rutas

In [ ]:
import os, torch

# === Modelo de CUENTOS (Llama + LoRA) ===
MODELO_TEXTO     = "meta-llama/Llama-3.1-8B-Instruct"
RUTA_LORA_TEXTO  = "/content/drive/MyDrive/cuentos_modelo/llama_lora/mejor"   # <-- ajusta si es otra ruta
USAR_QLORA_TEXTO = True

SYSTEM = ("Eres un escritor de cuentos infantiles en español. Escribe cuentos originales, "
          "claros, con inicio, desarrollo y final, apropiados para ninos.")

# === Modelo de ILUSTRACIONES (SD 1.5 + LoRA) ===
MODELO_IMG       = "stable-diffusion-v1-5/stable-diffusion-v1-5"
RUTA_LORA_IMG    = "/content/drive/MyDrive/ilustraciones_modelo/sd15_estilo_lora/final"   # <-- ajusta si es otra

NEG_PROMPT = "fotografia, realista, 3d, texto, marca de agua, deforme"

print("LoRA texto :", RUTA_LORA_TEXTO, "| existe:", os.path.isdir(RUTA_LORA_TEXTO))
print("LoRA imagen:", RUTA_LORA_IMG,   "| existe:", os.path.isdir(RUTA_LORA_IMG))


## Paso 2 — Cargar modelo de cuentos (Llama + LoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("Cargando Llama...")
tok_texto = AutoTokenizer.from_pretrained(RUTA_LORA_TEXTO)
if tok_texto.pad_token is None:
    tok_texto.pad_token = tok_texto.eos_token

if USAR_QLORA_TEXTO:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    base_texto = AutoModelForCausalLM.from_pretrained(MODELO_TEXTO, quantization_config=bnb,
                                                      torch_dtype=torch.bfloat16, device_map="auto")
else:
    base_texto = AutoModelForCausalLM.from_pretrained(MODELO_TEXTO, torch_dtype=torch.bfloat16, device_map="auto")

mod_texto = PeftModel.from_pretrained(base_texto, RUTA_LORA_TEXTO).eval()
print("Llama cargado.")


## Paso 3 — Cargar modelo de ilustraciones (SD 1.5 + LoRA)

In [ ]:
from diffusers import StableDiffusionPipeline
from peft import PeftModel as PeftModelSD

print("Cargando SD 1.5 + LoRA...")
pipe_img = StableDiffusionPipeline.from_pretrained(MODELO_IMG, torch_dtype=torch.float16, safety_checker=None)
pipe_img.unet = PeftModelSD.from_pretrained(pipe_img.unet, RUTA_LORA_IMG)
pipe_img = pipe_img.to("cuda")
print("SD 1.5 + LoRA cargado.")


## Paso 4 — Funciones del pipeline

Tres funciones:
1. `generar_cuento(tematica)` → texto del cuento.
2. `extraer_escena(cuento)` → frase visual corta (usa el mismo Llama, casi gratis).
3. `generar_ilustracion(tematica, escena)` → imagen con estilo del equipo + relevancia al cuento.


In [ ]:
@torch.no_grad()
def generar_cuento(tematica, titulo=None, temperatura=0.7, max_new=900):
    """Genera un cuento infantil condicionado por tematica."""
    tema = tematica.replace("_", " ").strip()
    if titulo:
        user = f'Escribe un cuento infantil titulado "{titulo}" sobre {tema}.'
    else:
        user = f"Escribe un cuento infantil sobre {tema}."
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]
    text = tok_texto.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok_texto(text, return_tensors="pt").to(mod_texto.device)
    out = mod_texto.generate(**inp, max_new_tokens=max_new, do_sample=True,
                             temperature=temperatura, top_p=0.8, top_k=20, repetition_penalty=1.1)
    return tok_texto.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

@torch.no_grad()
def extraer_escena(cuento, tematica):
    """Extrae UNA frase visual corta del cuento para condicionar la ilustracion."""
    tema = tematica.replace("_", " ").strip()
    msgs = [
        {"role": "system", "content": "Eres un asistente que describe escenas para ilustraciones infantiles."},
        {"role": "user", "content": (
            f"Del siguiente cuento sobre {tema}, describe en UNA sola frase corta (maximo 15 palabras) "
            f"la escena principal que se deberia ilustrar. Solo la frase, sin explicaciones.\n\n"
            f"Cuento:\n{cuento[:1500]}"
        )}
    ]
    text = tok_texto.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok_texto(text, return_tensors="pt").to(mod_texto.device)
    out = mod_texto.generate(**inp, max_new_tokens=40, do_sample=False, temperature=0.3)
    escena = tok_texto.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()
    # limpiar: quitar comillas, puntos al final, prefijos tipo "La escena es:"
    escena = escena.strip('"\'').split("\n")[0].strip()
    return escena

def generar_ilustracion(tematica, escena=None, pasos=30, guidance=7.5):
    """Genera una ilustracion con estilo LoRA + relevancia al cuento."""
    tema = tematica.replace("_", " ").strip()
    prompt = f"ilustracion plana infantil a color sobre {tema}"
    if escena:
        prompt += f", {escena}, estilo caricatura infantil"
    return pipe_img(prompt=prompt, negative_prompt=NEG_PROMPT,
                    num_inference_steps=pasos, guidance_scale=guidance).images[0]

print("Pipeline listo.")


## Paso 5 — Generar cuento ilustrado

Dale una **temática** y el pipeline hace todo: genera el cuento, extrae la escena visual,
y produce la ilustración. También puedes darle un **título** para dirigir el cuento.


In [ ]:
import matplotlib.pyplot as plt
from textwrap import fill
from IPython.display import display, HTML

def cuento_ilustrado(tematica, titulo=None):
    print(f"Generando cuento sobre '{tematica}'...")
    cuento = generar_cuento(tematica, titulo=titulo)

    print("Extrayendo escena visual...")
    escena = extraer_escena(cuento, tematica)
    print(f"  Escena: {escena}")

    print("Generando ilustracion...")
    imagen = generar_ilustracion(tematica, escena=escena)

    # Mostrar todo junto
    tema_display = tematica.replace("_", " ").title()
    fig, (ax_img, ax_txt) = plt.subplots(1, 2, figsize=(16, 8),
                                          gridspec_kw={"width_ratios": [1, 1.5]})
    ax_img.imshow(imagen)
    ax_img.set_title(f"Ilustracion: {tema_display}", fontsize=12, fontweight="bold")
    ax_img.axis("off")

    # Texto del cuento (ajustado al ancho)
    cuento_wrap = fill(cuento, width=55)
    ax_txt.text(0.02, 0.98, cuento_wrap, transform=ax_txt.transAxes, fontsize=7,
                verticalalignment="top", fontfamily="serif", wrap=True)
    ax_txt.set_title(f"Cuento: {tema_display}" + (f' — "{titulo}"' if titulo else ""),
                     fontsize=12, fontweight="bold")
    ax_txt.axis("off")
    plt.tight_layout()
    plt.show()

    return cuento, escena, imagen

# === Generar uno de prueba ===
cuento, escena, img = cuento_ilustrado("piratas")


## Paso 6 — Generar varios cuentos ilustrados

In [ ]:
temas = ["robots_y_tecnologia", "dinosaurios_y_prehistoria", "naturaleza_y_bosques", "espacio"]

resultados = []
for tema in temas:
    c, e, i = cuento_ilustrado(tema)
    resultados.append({"tematica": tema, "cuento": c, "escena": e, "imagen": i})
    print()


## Paso 7 — (Opcional) Guardar los cuentos ilustrados a Drive

In [ ]:
import json
from PIL import Image as PILImage

RUTA_RESULTADOS = "/content/drive/MyDrive/cuentos_ilustrados_generados"
os.makedirs(RUTA_RESULTADOS, exist_ok=True)

for r in resultados:
    tema = r["tematica"]
    # Guardar imagen
    r["imagen"].save(os.path.join(RUTA_RESULTADOS, f"{tema}_ilustracion.png"))
    # Guardar cuento + escena como JSON
    with open(os.path.join(RUTA_RESULTADOS, f"{tema}_cuento.json"), "w", encoding="utf-8") as f:
        json.dump({"tematica": tema, "cuento": r["cuento"], "escena": r["escena"]},
                  f, ensure_ascii=False, indent=2)

print(f"Guardados {len(resultados)} cuentos ilustrados en {RUTA_RESULTADOS}")
!ls -la {RUTA_RESULTADOS}
